# SuperSimpleNet


In [ ]:
import os
import sys
import re
import copy
import csv
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.optim as optim
import torch.nn.functional as torch_f
from torch.optim.lr_scheduler import MultiStepLR

from anomalib.models.image.supersimplenet.loss import SSNLoss
from anomalib.models.image.supersimplenet.torch_model import SupersimplenetModel

from sklearn.metrics import average_precision_score, roc_auc_score


In [ ]:
# CUDA/kernel guard. Run before training to make sure this notebook uses my_env with CUDA PyTorch.
import sys
import torch

print("Python executable:", sys.executable)
print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))

assert "+cu128" in torch.__version__, "Expected CUDA 12.8 PyTorch wheel, e.g. torch==2.7.1+cu128"
assert torch.cuda.is_available(), "CUDA is not visible from this notebook kernel"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)


# Конфигурация


In [ ]:
dataset_path = "datasets/processed_printer_dataset"
image_size = 256  # Same as SuperSimpleNet_v3; 320 can be tested later if needed.

# Cap tiles per date so quick experiments do not take hours.
# Set to None to train on the full dataset.
max_train_images_per_date = 200
max_val_images_per_date = 100
num_train_epochs = 30

logs_folder = "3D_printer_supersimplenet"
try_number = 2
log_dir = os.path.join("../experiments", logs_folder, f"try_{try_number}")


# Преобразования


In [ ]:
def get_augmented_transformer(image_size):
    return transforms.Compose([
        transforms.RandomResizedCrop((image_size, image_size), scale=(0.9, 1.1), ratio=(1, 1)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


def get_transformer(image_size):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


def postprocess_anomaly_map(anomaly_map, true_img_size):
    while anomaly_map.ndim < 4:
        anomaly_map = anomaly_map.unsqueeze(0)

    upsampled_mask = torch_f.interpolate(
        anomaly_map,
        size=true_img_size,
        mode="bilinear",
        align_corners=False
    )
    return upsampled_mask.squeeze()


# Датасеты


In [ ]:
class PrinterTrainDataset(Dataset):
    def __init__(self, root_dir, transform=None, include_dates=None, exclude_dates=None, max_images_per_date=None, seed=42):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.dates = []

        include_dates = set(include_dates) if include_dates is not None else None
        exclude_dates = set(exclude_dates) if exclude_dates is not None else set()

        for date_dir in sorted(os.listdir(self.root_dir)):
            if include_dates is not None and date_dir not in include_dates:
                continue
            if date_dir in exclude_dates:
                continue

            objects_dir = os.path.join(self.root_dir, date_dir, "objects_parts")
            if os.path.isdir(objects_dir):
                date_image_paths = []
                for img_name in sorted(os.listdir(objects_dir)):
                    if img_name.lower().endswith((".png", ".jpg", ".jpeg")):
                        date_image_paths.append(os.path.join(objects_dir, img_name))

                if max_images_per_date is not None and len(date_image_paths) > max_images_per_date:
                    rng = random.Random(f"{seed}_{date_dir}")
                    date_image_paths = sorted(rng.sample(date_image_paths, max_images_per_date))

                self.image_paths.extend(date_image_paths)
                self.dates.extend([date_dir] * len(date_image_paths))

        print(f"Loaded {len(self.image_paths)} object fragments from {self.root_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image


def anomaly_index_from_name(name):
    match = re.fullmatch(r"anomaly_(\d+)\.(png|jpg|jpeg)", name.lower())
    return int(match.group(1)) if match else None


def is_good_visual_image(name):
    return re.fullmatch(r"good_\d+\.(png|jpg|jpeg)", name.lower()) is not None


def is_visual_test_anomaly(name, max_index=16):
    anomaly_index = anomaly_index_from_name(name)
    return anomaly_index is not None and anomaly_index <= max_index


def is_head_finetune_anomaly(name, min_index=17):
    anomaly_index = anomaly_index_from_name(name)
    return anomaly_index is not None and anomaly_index >= min_index


def is_visual_test_image(name):
    return is_good_visual_image(name) or is_visual_test_anomaly(name)


def visual_test_label_from_name(name):
    if is_good_visual_image(name):
        return 0
    if is_visual_test_anomaly(name):
        return 1
    raise ValueError(f"{name} is not a visual-test image")


class ImageFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None, return_names=False, recursive=False, filename_filter=None):
        self.root_dir = root_dir
        self.transform = transform
        self.return_names = return_names
        self.image_paths = []

        walker = os.walk(root_dir) if recursive else [(root_dir, [], os.listdir(root_dir))]
        for current_dir, _, files in walker:
            for img_name in sorted(files):
                if not img_name.lower().endswith((".png", ".jpg", ".jpeg")):
                    continue
                if filename_filter is not None and not filename_filter(img_name):
                    continue
                self.image_paths.append(os.path.join(current_dir, img_name))

        print(f"Loaded {len(self.image_paths)} images from {self.root_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        if self.return_names:
            return image, os.path.basename(self.image_paths[idx])
        return image


class LabeledImageFolderDataset(ImageFolderDataset):
    def __init__(self, root_dir, transform=None, return_names=False, recursive=False, filename_filter=None, label_fn=None):
        super().__init__(root_dir, transform=transform, return_names=False, recursive=recursive, filename_filter=filename_filter)
        self.return_names = return_names
        self.label_fn = label_fn

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        name = os.path.basename(self.image_paths[idx])
        label = self.label_fn(name) if self.label_fn else 0
        if self.return_names:
            return image, label, name
        return image, label


def get_date_split(root_dir, validation_fraction=0.2):
    dates = []
    for date_dir in sorted(os.listdir(root_dir)):
        objects_dir = os.path.join(root_dir, date_dir, "objects_parts")
        if os.path.isdir(objects_dir):
            dates.append(date_dir)

    val_count = max(1, int(round(len(dates) * validation_fraction)))
    val_dates = dates[-val_count:]
    train_dates = dates[:-val_count]
    return train_dates, val_dates


In [ ]:
train_root = os.path.join(dataset_path, "training")
visual_test_dir = os.path.join(dataset_path, "visual_test_images")
train_dates, val_dates = get_date_split(train_root, validation_fraction=0.2)
print(f"Train dates: {train_dates}")
print(f"Validation dates: {val_dates}")

train_dataset = PrinterTrainDataset(
    train_root,
    transform=get_transformer(image_size),
    include_dates=train_dates,
    max_images_per_date=max_train_images_per_date,
    seed=random_seed,
)

normal_val_dataset = PrinterTrainDataset(
    train_root,
    transform=get_transformer(image_size),
    include_dates=val_dates,
    max_images_per_date=max_val_images_per_date,
    seed=random_seed,
)

visualization_dataset = ImageFolderDataset(
    visual_test_dir,
    transform=get_transformer(image_size),
    return_names=True,
    filename_filter=is_visual_test_image,
)

visual_test_classification_dataset = LabeledImageFolderDataset(
    visual_test_dir,
    transform=get_transformer(image_size),
    return_names=True,
    filename_filter=is_visual_test_image,
    label_fn=visual_test_label_from_name,
)

anomalies_dataset = ImageFolderDataset(
    visual_test_dir,
    transform=get_transformer(image_size),
    return_names=True,
    filename_filter=is_visual_test_anomaly,
)

head_finetune_anomaly_dataset = ImageFolderDataset(
    visual_test_dir,
    transform=get_transformer(image_size),
    return_names=True,
    filename_filter=is_head_finetune_anomaly,
)

print(f"Visual test images: {len(visualization_dataset)}")
print(f"Held-out visual anomalies for test: {len(anomalies_dataset)}")
print(f"Anomalies reserved for classifier-head fine-tune: {len(head_finetune_anomaly_dataset)}")


# Визуализация и метрики


In [ ]:
def img2mask(img):
    mask = img if img.ndim <= 2 else img.squeeze()
    if mask.ndim == 3:
        mask = mask[0, :, :]
    if mask.ndim != 2:
        raise ValueError("Wrong dimensions")
    return mask


def predict_label(pred_anomaly_map, k=0.001):
    pred_anomaly_map = img2mask(pred_anomaly_map)
    y_scores = pred_anomaly_map.flatten().cpu()
    top_k = max(1, int(k * len(y_scores)))
    return torch.mean(torch.topk(y_scores, top_k).values)


def predict_image_score(model, img, apply_sigmoid=False):
    model.eval()
    with torch.no_grad():
        output = model(img)
        raw_pred_score = output.pred_score.detach().view(-1)[0]
        if apply_sigmoid:
            return torch.sigmoid(raw_pred_score).item()
        return raw_pred_score.item()


def predict_anomaly_map(model, img, true_img_size=None, apply_sigmoid=True):
    model.eval()
    with torch.no_grad():
        output = model(img)
        anomaly_map = output.anomaly_map
        if apply_sigmoid:
            anomaly_map = torch.sigmoid(anomaly_map)
        if true_img_size is not None and anomaly_map.shape[-2:] != tuple(true_img_size):
            anomaly_map = postprocess_anomaly_map(anomaly_map, true_img_size=true_img_size)
    return anomaly_map




def ssn_training_forward(model, images, masks=None, labels=None):
    if masks is None:
        masks = torch.zeros((images.shape[0], images.shape[-2], images.shape[-1]), device=images.device)
    if labels is None:
        labels = torch.zeros((images.shape[0],), device=images.device)

    features = model.feature_extractor(images)
    adapted = model.adaptor(features)
    target_mask = model.downsample_mask(masks, *features.shape[-2:])
    target_label = labels.to(torch.float32)
    train_features, target_mask, target_label = model.anomaly_generator(adapted, target_mask, target_label)
    pred_map, pred_score = model.segdec(train_features)
    return pred_map, pred_score, target_mask, target_label


def plot_training_curves(metrics, metric_names, log_dir, plot_name, log_y=False):
    os.makedirs(log_dir, exist_ok=True)
    plt.figure(figsize=(10, 6))
    for metric, metric_name in zip(metrics, metric_names):
        plt.plot(metric, label=metric_name, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Metric")
    if log_y:
        plt.yscale("log")
    plt.title("Training metrics")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(log_dir, plot_name))
    plt.show()


def estimate_foreground_mask(image_tensor, min_coverage=0.02, max_coverage=0.98):
    image_tensor = torch.clamp(image_tensor.detach().cpu(), 0.0, 1.0)
    gray = image_tensor.mean(dim=0)
    threshold = max(0.05, float(gray.mean()) * 0.5)
    foreground = (gray > threshold).float()
    coverage = float(foreground.mean())
    if coverage < min_coverage or coverage > max_coverage:
        foreground = torch.ones_like(gray)
    return foreground


def foreground_anomaly_stats(anomaly_map, foreground_mask):
    anomaly = torch.as_tensor(anomaly_map, dtype=torch.float32).detach().cpu()
    foreground = torch.as_tensor(foreground_mask, dtype=torch.float32).detach().cpu()
    if foreground.shape != anomaly.shape:
        foreground = torch_f.interpolate(
            foreground[None, None],
            size=anomaly.shape,
            mode="nearest",
        ).squeeze()
    inside = anomaly[foreground > 0.5]
    outside = anomaly[foreground <= 0.5]
    return {
        "inside_mean": float(inside.mean()) if inside.numel() else 0.0,
        "inside_max": float(inside.max()) if inside.numel() else 0.0,
        "outside_mean": float(outside.mean()) if outside.numel() else 0.0,
        "outside_max": float(outside.max()) if outside.numel() else 0.0,
    }


def visualize_results(original_image, anomaly_map, save_path=None, score=None, figsize=None, alpha=0.7, cmap="jet"):
    denormalize = transforms.Compose([
        transforms.Normalize(mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225], std=[1/0.229, 1/0.224, 1/0.225])
    ])

    original_image = denormalize(original_image.squeeze().detach().cpu())
    original_image = torch.clamp(original_image, 0, 1)
    foreground_mask = estimate_foreground_mask(original_image)
    anomaly_map = img2mask(anomaly_map.detach().cpu())
    stats = foreground_anomaly_stats(anomaly_map, foreground_mask)
    masked_anomaly_map = anomaly_map * foreground_mask

    original_pil = transforms.ToPILImage()(original_image)
    anomaly_map = anomaly_map.numpy()
    masked_anomaly_map = masked_anomaly_map.numpy()

    if figsize is None:
        figsize = (15, 4)

    fig, axes = plt.subplots(1, 3, figsize=figsize)
    axes[0].imshow(original_pil)
    axes[0].set_title("Original image")
    axes[0].axis("off")

    axes[1].imshow(original_pil)
    heatmap = axes[1].imshow(anomaly_map, cmap=cmap, alpha=alpha, vmin=0.0, vmax=1.0)
    title = "Input + Anomaly Map"
    if score is not None:
        title += f" | score={score:.4f}"
    axes[1].set_title(title)
    axes[1].axis("off")
    plt.colorbar(heatmap, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].imshow(original_pil)
    masked_heatmap = axes[2].imshow(masked_anomaly_map, cmap=cmap, alpha=alpha, vmin=0.0, vmax=1.0)
    axes[2].set_title(
        f"Foreground diagnostic | in max={stats['inside_max']:.3f}, out max={stats['outside_max']:.3f}"
    )
    axes[2].axis("off")
    plt.colorbar(masked_heatmap, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path)
        plt.close(fig)
    else:
        plt.show()


def sigmoid_score(logit):
    return torch.sigmoid(torch.as_tensor(logit)).item()


def evaluate_visual_test(model, dataset, log_dir, image_size, device=device, eval_subdir="final_eval"):
    final_eval_dir = os.path.join(log_dir, eval_subdir)
    visualizations_dir = os.path.join(final_eval_dir, "visualizations")
    os.makedirs(visualizations_dir, exist_ok=True)

    model.eval()
    rows = []
    true_labels = []
    raw_pred_scores = []

    with torch.no_grad():
        for i in tqdm(range(len(dataset)), desc="Final Visual Test", leave=False):
            img, true_label, name = dataset[i]
            img = img.unsqueeze(0).to(device)

            raw_score = predict_image_score(model, img, apply_sigmoid=False)
            sigmoid_pred_score = sigmoid_score(raw_score)
            anomaly_map = predict_anomaly_map(model, img, true_img_size=(image_size, image_size), apply_sigmoid=True)

            visualize_results(
                img,
                anomaly_map,
                save_path=os.path.join(visualizations_dir, name),
                score=sigmoid_pred_score,
            )

            row = {
                "name": name,
                "label": int(true_label),
                "raw_pred_score": float(raw_score),
                "sigmoid_pred_score": float(sigmoid_pred_score),
            }
            rows.append(row)
            true_labels.append(int(true_label))
            raw_pred_scores.append(float(raw_score))

    image_roc_auc_raw = float(roc_auc_score(true_labels, raw_pred_scores))
    image_avg_precision_raw = float(average_precision_score(true_labels, raw_pred_scores))
    metrics = {
        "image_roc_auc_raw": image_roc_auc_raw,
        "image_avg_precision_raw": image_avg_precision_raw,
        # Backward-compatible aliases for older downstream cells.
        "image_roc_auc": image_roc_auc_raw,
        "image_avg_precision": image_avg_precision_raw,
        "num_images": len(rows),
        "num_good": int(sum(1 for label in true_labels if label == 0)),
        "num_anomaly": int(sum(1 for label in true_labels if label == 1)),
    }

    with open(os.path.join(final_eval_dir, "per_image_scores.csv"), "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["name", "label", "raw_pred_score", "sigmoid_pred_score"])
        writer.writeheader()
        writer.writerows(rows)

    with open(os.path.join(final_eval_dir, "classification_metrics.txt"), "w") as f:
        for key, value in metrics.items():
            f.write(f"{key}: {value}\n")

    return metrics, rows


# Обучение модели


In [ ]:
def freeze_feature_extractor(model):
    frozen_params_num = 0
    if hasattr(model, "feature_extractor"):
        for param in model.feature_extractor.parameters():
            param.requires_grad = False
            frozen_params_num += param.numel()
    print(f"Parameters frozen: {frozen_params_num}")


def get_trainable_parameters(model):
    trainable_params = []
    train_params_num = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            trainable_params.append(param)
            train_params_num += param.numel()
    print(f"Trainable parameters: {train_params_num}")
    return trainable_params


In [ ]:
def train_ssn_with_validation(
    model,
    train_dataset,
    validation_dataset=None,
    visualization_dataset=None,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=1e-3,
    device=device,
    log_interval=1,
    log_dir="../experiments",
    freeze_extractor=True,
    batch_size=8,
    num_workers=0,
    resume_from_checkpoint=False,
    early_stopping_patience=5,
    early_stopping_metric="Validation_Loss",
    early_stopping_mode="min",
):
    model = model.to(device)
    criterion = SSNLoss()
    os.makedirs(log_dir, exist_ok=True)
    checkpoints_dir = os.path.join(log_dir, "middle_logs")
    os.makedirs(checkpoints_dir, exist_ok=True)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters number: {total_params}")
    if freeze_extractor:
        freeze_feature_extractor(model)
    get_trainable_parameters(model)

    optimizer = optim.AdamW([
        {"params": model.adaptor.parameters(), "lr": learning_rate},
        {"params": model.segdec.parameters(), "lr": learning_rate * 2, "weight_decay": weight_decay},
    ])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = None
    if validation_dataset is not None and len(validation_dataset) > 0:
        val_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    scheduler = MultiStepLR(
        optimizer,
        milestones=[max(1, int(num_epochs * 0.8)), max(1, int(num_epochs * 0.9))],
        gamma=0.4,
    )

    best_score = -float("inf") if early_stopping_mode == "max" else float("inf")
    epochs_without_improvement = 0
    best_model_state = None
    best_epoch = None
    checkpoint_path = os.path.join(checkpoints_dir, "checkpoint.pth")
    start_epoch = 0
    epoch_scores = {"Train_Loss": []}
    if val_loader is not None:
        epoch_scores["Validation_Loss"] = []

    if resume_from_checkpoint and os.path.exists(checkpoint_path):
        print(f"Loading checkpoint from {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        if "scheduler_state_dict" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        start_epoch = checkpoint["epoch"]
        epoch_scores = checkpoint["epoch_scores"]
        print(f"Resuming from epoch {start_epoch + 1}")
    else:
        print("Starting training from scratch...")

    for epoch in range(start_epoch, num_epochs):
        model.train()
        total_loss = 0.0
        for images in tqdm(train_loader, desc=f"Train Epoch {epoch+1}/{num_epochs}", leave=False):
            images = images.to(device)
            optimizer.zero_grad()
            pred_map, pred_score, target_mask, target_label = ssn_training_forward(model, images)
            loss = criterion(pred_map=pred_map, pred_score=pred_score, target_mask=target_mask, target_label=target_label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        epoch_scores["Train_Loss"].append(avg_train_loss)
        scheduler.step()

        if val_loader is not None:
            model.eval()
            total_val_loss = 0.0
            with torch.no_grad():
                for images in tqdm(val_loader, desc=f"Val Epoch {epoch+1}/{num_epochs}", leave=False):
                    images = images.to(device)
                    pred_map, pred_score, target_mask, target_label = ssn_training_forward(model, images)
                    loss = criterion(pred_map=pred_map, pred_score=pred_score, target_mask=target_mask, target_label=target_label)
                    total_val_loss += loss.item()
            avg_val_loss = total_val_loss / len(val_loader)
            epoch_scores["Validation_Loss"].append(avg_val_loss)
        else:
            avg_val_loss = None

        if (epoch + 1) % log_interval == 0:
            if avg_val_loss is None:
                print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.6f}")
            else:
                print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

            if visualization_dataset is not None:
                model.eval()
                visualization_dir = os.path.join(checkpoints_dir, "visualization", f"epoch_{epoch + 1}")
                with torch.no_grad():
                    for i in range(len(visualization_dataset)):
                        img, name = visualization_dataset[i]
                        img = img.unsqueeze(0).to(device)
                        anomaly_map = predict_anomaly_map(model, img, true_img_size=(image_size, image_size), apply_sigmoid=True)
                        score = predict_image_score(model, img, apply_sigmoid=True)
                        visualize_results(img, anomaly_map, save_path=os.path.join(visualization_dir, name), score=score)

        checkpoint_payload = {
            "epoch": epoch + 1,
            "model_state_dict": copy.deepcopy(model.state_dict()),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "epoch_scores": epoch_scores,
            "learning_rate": learning_rate,
        }
        torch.save(checkpoint_payload, checkpoint_path)
        torch.save(checkpoint_payload, os.path.join(checkpoints_dir, f"checkpoint_epoch_{epoch + 1}.pth"))

        if early_stopping_metric in epoch_scores:
            current_score = epoch_scores[early_stopping_metric][-1]
            is_better = (
                (early_stopping_mode == "max" and current_score > best_score) or
                (early_stopping_mode == "min" and current_score < best_score)
            )
            if is_better:
                best_score = current_score
                epochs_without_improvement = 0
                best_model_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch + 1
                print(f"New best {early_stopping_metric}: {best_score:.6f}")
            else:
                epochs_without_improvement += 1
                print(f"No improvement for {epochs_without_improvement}/{early_stopping_patience} epochs")

            if epochs_without_improvement >= early_stopping_patience:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Loaded best model from epoch {best_epoch} (score: {best_score:.6f})")
    else:
        print("No early stopping metric found - using final model")

    torch.save(model.state_dict(), os.path.join(log_dir, "trained_model_weights.pth"))
    torch.save(model, os.path.join(log_dir, "trained_model.pth"))

    with open(os.path.join(checkpoints_dir, "training_metrics.txt"), "w") as f:
        for metric_name, metric in epoch_scores.items():
            f.write(f"{metric_name}: {metric[-1]}\n")
        if best_epoch is not None:
            f.write(f"best_epoch: {best_epoch}\n")
            f.write(f"best_{early_stopping_metric}: {best_score}\n")

    plot_training_curves(
        [metric for metric in epoch_scores.values()],
        [metric_name for metric_name in epoch_scores.keys()],
        checkpoints_dir,
        "training_metrics.png",
    )
    return model


In [ ]:
model = SupersimplenetModel(
    backbone="wide_resnet50_2.tv_in1k",
    layers=["layer2", "layer3"],
    perlin_threshold=0.2,
    stop_grad=True,
)

trained_model = train_ssn_with_validation(
    model=model,
    train_dataset=train_dataset,
    validation_dataset=normal_val_dataset,
    visualization_dataset=visualization_dataset,
    num_epochs=num_train_epochs,
    learning_rate=1e-4,
    weight_decay=1e-5,
    device=device,
    log_interval=3,
    log_dir=log_dir,
    batch_size=10,
    resume_from_checkpoint=False,
    freeze_extractor=True,
    early_stopping_patience=6,
    early_stopping_metric="Validation_Loss",
    early_stopping_mode="min",
)


# Проверка на аномальных примерах


In [ ]:
best_model_path = os.path.join(log_dir, "trained_model.pth")
best_trained_model = torch.load(best_model_path, weights_only=False, map_location=device)
best_trained_model.to(device)
best_trained_model.eval()

final_eval_metrics, final_eval_rows = evaluate_visual_test(
    best_trained_model,
    visual_test_classification_dataset,
    log_dir=log_dir,
    image_size=image_size,
    device=device,
)

print(f"Final image ROC AUC: {final_eval_metrics['image_roc_auc']:.4f}")
print(f"Final image average precision: {final_eval_metrics['image_avg_precision']:.4f}")
print(f"Final visualizations saved to: {os.path.join(log_dir, 'final_eval', 'visualizations')}")


In [ ]:
# Final visual-test artifacts are saved in log_dir/final_eval.
